# Python Call

In [ ]:
from kfp import dsl
from kfp.dsl import component
from kfp import compiler
from google.cloud import aiplatform

# --- Configuration ---
PROJECT_ID = "anbc-dev-hcm-cm-de"
LOCATION = "us-east4"
PIPELINE_ROOT = "gs://hcm-cm-de-code-hcb-dev/vertex-test/"
TRAINING_DATASET = "gs://hcm-cm-de-data-hcb-dev/zj_a455644_test/peft_train_sample_gemini.jsonl"
VALIDATION_DATASET = "gs://hcm-cm-de-data-hcb-dev/zj_a455644_test/peft_eval_sample_gemini.jsonl"
BASE_MODEL = "gemini-2.0-flash-001"  # Or gemini-1.5-pro-001
TUNED_MODEL_NAME = "gemini-2.0-flash-001-tuned-v1"

# --- Custom Component for Gemini Tuning ---
@component(
    base_image="python:3.9",
    packages_to_install=[
        "google-cloud-aiplatform>=1.38.0",
        "vertexai>=1.38.0"
    ]
)
def gemini_tuning_component(
    project: str,
    location: str,
    base_model: str,
    training_dataset_uri: str,
    validation_dataset_uri: str,
    tuned_model_display_name: str,
    service_account: str,
    epochs: int = 3,
    learning_rate_multiplier: float = 1.0,
    encryption_spec_key_name: str = None,
) -> str:
    """
    Custom component to create a Gemini supervised tuning job.
    Returns the tuned model resource name.
    """
    from vertexai.tuning import sft
    from google.cloud import aiplatform
    
    # Initialize Vertex AI
    aiplatform.init(project=project, location=location)
    
    # Create supervised tuning job
    tuning_job = sft.train(
        source_model=base_model,
        train_dataset=training_dataset_uri,
        validation_dataset=validation_dataset_uri,
        tuned_model_display_name=tuned_model_display_name,
        epochs=epochs,
        learning_rate_multiplier=learning_rate_multiplier
    )

    
    # Return the tuned model resource name
    return tuning_job.tuned_model_name

# --- Pipeline Definition ---
@dsl.pipeline(
    name="gemini-managed-tuning-pipeline",
    description="Pipeline to fine-tune Gemini Pro/Flash"
)
def pipeline(
    project_id: str = PROJECT_ID,
    location: str = LOCATION,
    base_model: str = BASE_MODEL,
    training_dataset: str = TRAINING_DATASET,
    validation_dataset: str = VALIDATION_DATASET,
    tuned_model_name: str = TUNED_MODEL_NAME,
    service_account: str = "gchcb-hcm-cm-de-ontpd@anbc-dev-hcm-cm-de.iam.gserviceaccount.com",
    epochs: int = 3,
    learning_rate_multiplier: float = 1.0,
    encryption_spec_key_name: str = "projects/cvs-key-vault-nonprod/locations/us-east4/keyRings/gkr-nonprod-us-east4/cryptoKeys/gk-anbc-dev-hcm-cm-de-us-east4"
):
    tuning_op = gemini_tuning_component(
        project=project_id,
        location=location,
        base_model=base_model,
        training_dataset_uri=training_dataset,
        validation_dataset_uri=validation_dataset,
        tuned_model_display_name=tuned_model_name,
        service_account=service_account,
        epochs=epochs,
        learning_rate_multiplier=learning_rate_multiplier,
        encryption_spec_key_name=encryption_spec_key_name
    )

    # --- Compile and Run ---
compiler.Compiler().compile(
    pipeline_func=pipeline, package_path="gemini_tuning_pipeline.yaml"
)

aiplatform.init(project=PROJECT_ID, location=LOCATION)

job = aiplatform.PipelineJob(
    display_name="gemini-tuning-run",
    template_path="gemini_tuning_pipeline.yaml",
    pipeline_root=PIPELINE_ROOT,
    enable_caching=False,
    encryption_spec_key_name="projects/cvs-key-vault-nonprod/locations/us-east4/keyRings/gkr-nonprod-us-east4/cryptoKeys/gk-anbc-dev-hcm-cm-de-us-east4",
    labels={
        "owner": "_aetna_com",
        "pipeline_type": "tuning",
        "lob": "hcb",
        "costcenter": "13070",
        "tenant": "hcm-cm-de",
        "self_serve": "true"
    }
)


In [ ]:
job.run(
    service_account="gchcb-hcm-cm-de-ontpd@anbc-dev-hcm-cm-de.iam.gserviceaccount.com",
    sync=True
)

# REST API CALL

In [ ]:
from kfp import dsl
from kfp.dsl import component
from kfp import compiler
from google.cloud import aiplatform
import json
import time

# --- Configuration ---
PROJECT_ID = "anbc-dev-hcm-cm-de"
LOCATION = "us-east4"
PIPELINE_ROOT = "gs://hcm-cm-de-code-hcb-dev/vertex-test/"
TRAINING_DATASET = "gs://hcm-cm-de-data-hcb-dev/zj_a455644_test/peft_train_sample_gemini.jsonl"
VALIDATION_DATASET = "gs://hcm-cm-de-data-hcb-dev/zj_a455644_test/peft_eval_sample_gemini.jsonl"
BASE_MODEL = "gemini-2.0-flash-001"  # Or gemini-1.5-pro-001
TUNED_MODEL_NAME = "gemini-2.0-flash-001-tuned-v1"
SERVICE_ACCOUNT = "gchcb-hcm-cm-de-ontpd@anbc-dev-hcm-cm-de.iam.gserviceaccount.com"

# --- Custom Component for Gemini Tuning using REST API ---
@component(
    base_image="python:3.9",
    packages_to_install=[
        "google-cloud-aiplatform>=1.38.0",
        "google-auth>=2.0.0",
        "requests>=2.25.0"
    ]
)
def gemini_tuning_component(
    project: str,
    location: str,
    base_model: str,
    training_dataset_uri: str,
    validation_dataset_uri: str,
    tuned_model_display_name: str,
    service_account: str,
    epochs: int = 3,
    learning_rate_multiplier: float = 1.0,
    encryption_spec_key_name: str = None,
) -> str:
    """
    Custom component to create a Gemini supervised tuning job using REST API.
    Returns the tuned model resource name.
    """
    from google.cloud import aiplatform
    from google.cloud import bigquery
    import requests
    import json
    from google.auth import default
    from google.auth.transport.requests import Request
    
    # Get authentication credentials
    credentials, _ = default()
    credentials.refresh(Request())
    access_token = credentials.token
    
    # Construct the REST API endpoint
    api_url = f"https://{location}-aiplatform.googleapis.com/v1/projects/{project}/locations/{location}/tuningJobs"
    
    # Build the request body
    request_body = {
        "baseModel": base_model,
        "supervisedTuningSpec": {
            "trainingDatasetUri": training_dataset_uri,
            "validationDatasetUri": validation_dataset_uri,
            "hyperParameters": {
                "epochCount": str(epochs),
                "learningRateMultiplier": str(learning_rate_multiplier)
            }
        },
        "tunedModelDisplayName": tuned_model_display_name,
        "serviceAccount": service_account
    }
    
    # Add encryption spec if provided
    if encryption_spec_key_name:
        request_body["encryptionSpec"] = {
            "kmsKeyName": encryption_spec_key_name
        }
    
    # Make the REST API call
    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json"
    }
    
    print(f"Creating tuning job with REST API...")
    print(f"URL: {api_url}")
    print(f"Request body: {json.dumps(request_body, indent=2)}")
    
    response = requests.post(
        api_url,
        headers=headers,
        json=request_body
    )
    
    # Check response
    if response.status_code != 200:
        raise Exception(f"Failed to create tuning job: {response.status_code} - {response.text}")
    
    tuning_job = response.json()
    tuning_job_name = tuning_job.get("name")
    
    print(f"Tuning job created: {tuning_job_name}")
    
    # Wait for the tuning job to complete
    # Poll the job status
    job_status_url = f"https://{location}-aiplatform.googleapis.com/v1/{tuning_job_name}"
    
    while True:
        status_response = requests.get(
            job_status_url,
            headers=headers
        )
        
        if status_response.status_code != 200:
            raise Exception(f"Failed to get tuning job status: {status_response.status_code} - {status_response.text}")
        
        job_status = status_response.json()
        state = job_status.get("state", "UNKNOWN")
        
        print(f"Tuning job state: {state}")
        
        if state == "JOB_STATE_SUCCEEDED":
            # Extract the tuned model resource name
            tuned_model_name = job_status.get("tunedModel", "")
            if not tuned_model_name:
                raise Exception("Tuning job succeeded but no tuned model name found")
            print(f"Tuning completed. Tuned model: {tuned_model_name}")
            return tuned_model_name
        elif state in ["JOB_STATE_FAILED", "JOB_STATE_CANCELLED"]:
            error_msg = job_status.get("error", {}).get("message", "Unknown error")
            raise Exception(f"Tuning job failed: {error_msg}")
        
        # Wait before polling again
        time.sleep(60)  # Poll every minute

# --- Pipeline Definition ---
@dsl.pipeline(
    name="gemini-managed-tuning-pipeline",
    description="Pipeline to fine-tune Gemini Pro/Flash using REST API"
)
def pipeline(
    project_id: str = PROJECT_ID,
    location: str = LOCATION,
    base_model: str = BASE_MODEL,
    training_dataset: str = TRAINING_DATASET,
    validation_dataset: str = VALIDATION_DATASET,
    tuned_model_name: str = TUNED_MODEL_NAME,
    service_account: str = SERVICE_ACCOUNT,
    epochs: int = 3,
    learning_rate_multiplier: float = 1.0,
    encryption_spec_key_name: str = "projects/cvs-key-vault-nonprod/locations/us-east4/keyRings/gkr-nonprod-us-east4/cryptoKeys/gk-anbc-dev-hcm-cm-de-us-east4"
):
    tuning_op = gemini_tuning_component(
        project=project_id,
        location=location,
        base_model=base_model,
        training_dataset_uri=training_dataset,
        validation_dataset_uri=validation_dataset,
        tuned_model_display_name=tuned_model_name,
        service_account=service_account,
        epochs=epochs,
        learning_rate_multiplier=learning_rate_multiplier,
    )

# --- Compile and Run ---
compiler.Compiler().compile(
    pipeline_func=pipeline, package_path="gemini_tuning_pipeline.yaml"
)

aiplatform.init(project=PROJECT_ID, location=LOCATION)

job = aiplatform.PipelineJob(
    display_name="gemini-tuning-run",
    template_path="gemini_tuning_pipeline.yaml",
    pipeline_root=PIPELINE_ROOT,
    enable_caching=False,
    encryption_spec_key_name="projects/cvs-key-vault-nonprod/locations/us-east4/keyRings/gkr-nonprod-us-east4/cryptoKeys/gk-anbc-dev-hcm-cm-de-us-east4",
    labels={
        "owner": "sahil_gadge_aetna_com",
        "pipeline_type": "tuning",
        "lob": "hcb",
        "costcenter": "13070",
        "tenant": "hcm-cm-de",
        "self_serve": "true"
    }
)


In [ ]:

job.run(
    service_account="gchcb-hcm-cm-de-ontpd@anbc-dev-hcm-cm-de.iam.gserviceaccount.com",
    sync=True
)